<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/00-B_Dataset%20preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## ⭐ Setup

### Packages Handling

In [1]:
%pip install mrcfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.9 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import mrcfile
from sklearn.model_selection import train_test_split

### Function

In [3]:
def save_mrc_to_npy(filenames, mrc_dir, save_dir, permissive=False):
  for filename in filenames:
    filepath = os.path.join(mrc_dir, filename)
    print("\rReading mrc:", filepath, end="", flush=True)
    with mrcfile.open(filepath, permissive=permissive) as mrc:
      basename, ext = os.path.splitext(filename)
      assert ext == ".mrc"
      print("\rConverting mrc to npy.", end="", flush=True)
      np.save(os.path.join(save_dir, basename), mrc.data)

## ⭐ Main

In [7]:
# @title Setting directory for mrc files.
from google.colab import drive
drive.mount('/content/drive')

MRC_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/processed_micrographs" # @param {type:"string"}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# title Train test split
random_state = None # @param {type:"integer"}
test_size = "0.83" # @param {type:"string"}
val_size = "0.071" # @param {type:"string"}
filenames = sorted(os.listdir(MRC_DIR))

test_size = float(test_size)
if test_size and not 0<=test_size<1:
  raise ValueError(f"`test_size` should be between 0.0 and 1.0, got: {test_size}")
val_size = float(val_size)
if val_size and not 0<=val_size<1:
  raise ValueError(f"`val_size` should be between 0.0 and 1.0, got: {val_size}")
train_size = 1 - test_size - val_size
if not 0<train_size<=1:
  raise ValueError(f"`val_size` + `test_size` should be between 0.0 and 1.0, got: {test_size + val_size}")
train_filenames, test_filenames = train_test_split(filenames, test_size=test_size, train_size=train_size, random_state=3)
train_filenames = sorted(train_filenames)
test_filenames = sorted(test_filenames)
val_filenames = []
for filename in filenames:
  if filename not in train_filenames and filename not in test_filenames:
    val_filenames.append(filename)
print("number of files:",
  f" train:\t{len(train_filenames)}",
  f" test:\t{len(test_filenames)}",
  f" val:\t{len(val_filenames)}", sep='\n')

number of files:
 train:	8
 test:	70
 val:	6


Alternatively, you can directly specify the split by loading it from files.

In [9]:
SAVE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
!mkdir $SAVE_DIR

In [10]:
%%capture --no-display
# @title Generate np file from mrc files.
# @markdown note that directory for np files should not exist


PERMISSIVE = True # @param {type:"boolean"}

try:
    os.mkdir(SAVE_DIR)
except FileExistsError:
    pass

os.mkdir(os.path.join(SAVE_DIR, "train"))
save_mrc_to_npy(train_filenames,
                mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "train"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "train_filenames.txt"), train_filenames, fmt="%s")

os.mkdir(os.path.join(SAVE_DIR, "test"))
save_mrc_to_npy(test_filenames, mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "test"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "test_filenames.txt"), test_filenames, fmt="%s")

os.mkdir(os.path.join(SAVE_DIR, "val"))
save_mrc_to_npy(val_filenames, mrc_dir=MRC_DIR,
                save_dir=os.path.join(SAVE_DIR, "val"),
                permissive=PERMISSIVE)
np.savetxt(os.path.join(SAVE_DIR, "val_filenames.txt"), val_filenames, fmt="%s")

In [11]:
file_path = os.path.join(SAVE_DIR, 'train_filenames.txt')
file_path2 = os.path.join(SAVE_DIR, 'val_filenames.txt')
file_path3 = os.path.join(SAVE_DIR, 'test_filenames.txt')

with open(file_path, 'r') as f:
    train_filenames = [line.strip() for line in f if line.strip()]

with open(file_path2, 'r') as f:
    val_filenames = [line.strip() for line in f if line.strip()]

with open(file_path3, 'r') as f:
    test_filenames = [line.strip() for line in f if line.strip()]

print("number of files:",
  f" train:\t{len(train_filenames)}",
  f" test:\t{len(test_filenames)}",
  f" val:\t{len(val_filenames)}", sep='\n')

number of files:
 train:	8
 test:	70
 val:	6
